In [1]:
import ee
ee.Authenticate()
ee.Initialize(project="climateconsciousimli")

# Date range
start_date = '2023-01-01'
end_date = '2023-01-31'

# NO2 collection
no2_col = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
           .filterDate(start_date, end_date)
           .select('tropospheric_NO2_column_number_density'))

# CO collection
co_col = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_CO')
          .filterDate(start_date, end_date)
          .select('CO_column_number_density'))


In [52]:
lat_min, lat_max = 28.0, 29.39
lat_avg = (lat_max + lat_min) / 2
lon_min, lon_max = 76.3, 79.0
lon_avg = (lon_max + lon_min) / 2

stepp = 0.25

region = ee.Geometry.Rectangle([lon_avg-stepp, lat_avg-stepp, lon_avg+stepp, lat_avg+stepp])

scale = 20000  # e.g., 10 km (change as needed)

def sample_collection(img_col, band_name):
    def sample_image(img):
        date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd HH:mm:ss')
        
        sampled = img.sample(
            region=region,
            scale=scale,
            geometries=True
        )
        
        return sampled.map(lambda f: f.set({
            'datetime': date,
            'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
            'band': band_name
        }))
    
    return img_col.map(sample_image).flatten()

no2_samples = sample_collection(no2_col, 'NO2')
co_samples = sample_collection(co_col, 'CO')

In [53]:
awa = [no2_samples, co_samples]
print(
    awa[0].getInfo()['features'][1]['properties'].keys()
)
print()
print(
    awa[1].getInfo()['features'][1]['properties'].keys()
)

dict_keys(['band', 'date', 'datetime', 'tropospheric_NO2_column_number_density'])

dict_keys(['CO_column_number_density', 'band', 'date', 'datetime'])


In [54]:
import pandas as pd

def fc_to_df(fc):
    data = fc.getInfo()['features']
    rows = []
    for f in data:
        props = f['properties']
        coords = f['geometry']['coordinates']
        owow = list(props.values())[0]
        othow = list(props.values())[-1]
        if type(owow) is str :
            use_index = -1
        else :
            use_index = 0
        rows.append({
            'lon': coords[0],
            'lat': coords[1],
            'datetime': props['datetime'],
            'date': props['date'],
            props['band'] : list(props.values())[use_index]
  # band value
        })
    return pd.DataFrame(rows)

df_no2 = fc_to_df(no2_samples)
df_co = fc_to_df(co_samples)

# Merge on lat, lon, datetime
df = pd.merge(df_no2, df_co, on=['lat','lon','datetime','date'], how='outer')
df = df.sort_values(['lat','lon','datetime'])

In [56]:
df

,lon,lat,datetime,date,NO2,CO
0,77.546992,28.487786,2023-01-01 07:10:53,2023-01-01,0.000191,0.042295
1,77.546992,28.487786,2023-01-02 06:51:53,2023-01-02,NaN,0.040134
2,77.546992,28.487786,2023-01-03 06:32:54,2023-01-03,NaN,0.039963
3,77.546992,28.487786,2023-01-04 07:55:24,2023-01-04,0.000227,0.045353
4,77.546992,28.487786,2023-01-05 07:36:26,2023-01-05,0.000346,0.047284
...,...,...,...,...,...,...
162,77.726655,28.847112,2023-01-22 07:18:26,2023-01-22,0.000051,0.043086
163,77.726655,28.847112,2023-01-23 06:59:28,2023-01-23,0.000070,0.032667
164,77.726655,28.847112,2023-01-26 07:44:07,2023-01-26,NaN,0.037802
165,77.726655,28.847112,2023-01-27 07:25:10,2023-01-27,0.000065,0.039453


In [57]:
df['NO2'] = pd.to_numeric(df['NO2'], errors='coerce')
df['CO'] = pd.to_numeric(df['CO'], errors='coerce')

df['datetime'] = pd.to_datetime(df['datetime'])
df['date'] = pd.to_datetime(df['date'])
df.set_index('date')

df = df.sort_values(['lat','lon','datetime'])

df[['NO2','CO']] = (
    df.set_index('datetime')
      .groupby(['lat','lon'])[['NO2','CO']]
      .apply(lambda g: g.interpolate(method='time'))
      .reset_index(level=[0,1], drop=True)
      .reset_index(drop=True)
)

In [58]:
df

,lon,lat,datetime,date,NO2,CO
0,77.546992,28.487786,2023-01-01 07:10:53,2023-01-01,0.000191,0.042295
1,77.546992,28.487786,2023-01-02 06:51:53,2023-01-02,0.000202,0.040134
2,77.546992,28.487786,2023-01-03 06:32:54,2023-01-03,0.000214,0.039963
3,77.546992,28.487786,2023-01-04 07:55:24,2023-01-04,0.000227,0.045353
4,77.546992,28.487786,2023-01-05 07:36:26,2023-01-05,0.000346,0.047284
...,...,...,...,...,...,...
162,77.726655,28.847112,2023-01-22 07:18:26,2023-01-22,0.000051,0.043086
163,77.726655,28.847112,2023-01-23 06:59:28,2023-01-23,0.000070,0.032667
164,77.726655,28.847112,2023-01-26 07:44:07,2023-01-26,0.000066,0.037802
165,77.726655,28.847112,2023-01-27 07:25:10,2023-01-27,0.000065,0.039453


In [59]:
df = df.sort_values(['lat','lon','datetime'])

def create_lags(group):
    group = group.sort_values('datetime')
    
    for i in range(1, 4):
        group[f'NO2_lag{i}'] = group['NO2'].shift(i)
        group[f'CO_lag{i}'] = group['CO'].shift(i)
    
    group['NO2_target'] = group['NO2'].shift(-1)
    group['CO_target'] = group['CO'].shift(-1)
    
    return group

df_model = df.groupby(['lat','lon']).apply(create_lags).reset_index(drop=True)

# Drop rows with missing values
df_model = df_model.dropna()

In [60]:
df_model

,datetime,date,NO2,CO,NO2_lag1,CO_lag1,NO2_lag2,CO_lag2,NO2_lag3,CO_lag3,NO2_target,CO_target
3,2023-01-04 07:55:24,2023-01-04,0.000227,0.045353,0.000214,0.039963,0.000202,0.040134,0.000191,0.042295,0.000346,0.047284
4,2023-01-05 07:36:26,2023-01-05,0.000346,0.047284,0.000227,0.045353,0.000214,0.039963,0.000202,0.040134,0.000381,0.056909
5,2023-01-06 07:17:26,2023-01-06,0.000381,0.056909,0.000346,0.047284,0.000227,0.045353,0.000214,0.039963,0.000408,0.048118
6,2023-01-07 06:58:28,2023-01-07,0.000408,0.048118,0.000381,0.056909,0.000346,0.047284,0.000227,0.045353,0.000404,0.045549
7,2023-01-08 06:39:29,2023-01-08,0.000404,0.045549,0.000408,0.048118,0.000381,0.056909,0.000346,0.047284,0.000519,0.047928
...,...,...,...,...,...,...,...,...,...,...,...,...
161,2023-01-21 07:37:24,2023-01-21,0.000130,0.045660,0.000096,0.038969,0.000223,0.041178,0.000119,0.040795,0.000051,0.043086
162,2023-01-22 07:18:26,2023-01-22,0.000051,0.043086,0.000130,0.045660,0.000096,0.038969,0.000223,0.041178,0.000070,0.032667
163,2023-01-23 06:59:28,2023-01-23,0.000070,0.032667,0.000051,0.043086,0.000130,0.045660,0.000096,0.038969,0.000066,0.037802
164,2023-01-26 07:44:07,2023-01-26,0.000066,0.037802,0.000070,0.032667,0.000051,0.043086,0.000130,0.045660,0.000065,0.039453


In [61]:
from sklearn.linear_model import LinearRegression

features = [
    'NO2_lag1','NO2_lag2','NO2_lag3',
    'CO_lag1','CO_lag2','CO_lag3'
]

X = df_model[features]
y_no2 = df_model['NO2_target']
y_co = df_model['CO_target']

model_no2 = LinearRegression()
model_co = LinearRegression()

model_no2.fit(X, y_no2)
model_co.fit(X, y_co)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [62]:
df_model['NO2_pred'] = model_no2.predict(X)
df_model['CO_pred'] = model_co.predict(X)